In [1]:
!pip install efficientnet-pytorch

  Preparing metadata (setup.py) ... done
  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16426 sha256=9c574b6f3d78f2d4926d43583681c16ac47e7089954eaf04d43bf552ba62b9a5
  Stored in directory: /root/.cache/pip/wheels/9c/3f/43/e6271c7026fe08c185da2be23c98c8e87477d3db63f41f32ad
Successfully built efficientnet-pytorch


In [6]:
import os
from pathlib import Path

BASE = Path("/kaggle/input/datasets/sanjeev1111/dataset")

print("BASE folders:")
print(os.listdir(BASE))

print("\n1K folders:")
print(os.listdir(BASE / "AGIQA-1k-Database-main/AGIQA-1k-Database-main"))

print("\n3K folders:")
print(os.listdir(BASE / "AGIQA-3k-Database-main/AGIQA-3k-Database-main"))

BASE folders:
['AGIQA-3k-Database-main', 'AGIQA-1k-Database-main']

1K folders:
['LICENSE', 'README.md', 'AIGC_MOS_Zscore.xlsx', 'AIGC_all.csv']

3K folders:
['LICENSE', 'README.md', 'data.csv']


In [9]:
# =========================================
# 🏆 HIGH-SRCC IQA PIPELINE
# =========================================

import warnings, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import lightgbm as lgb

warnings.filterwarnings("ignore")

# ===== CONFIG =====
EPOCHS   = 15
BATCH    = 16
LR       = 2e-4
N_SPLITS = 5          # more folds → better OOF coverage
SEEDS    = [42, 123, 999]
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ===== PATHS =====
BASE = Path("/kaggle/input/datasets/sanjeev1111/dataset")
IMGB = Path("/kaggle/input/datasets/sanjeev1111/image-data")

# ===== DATA =====
def build_map(folder):
    return {f.stem.lower(): str(f) for f in folder.iterdir()} | \
           {f.name.lower(): str(f) for f in folder.iterdir()}

def load_data():
    map1k = build_map(IMGB / "file/file")
    map3k = build_map(IMGB / "AGIQA-3K")

    df1 = pd.read_excel(BASE / "AGIQA-1k-Database-main/AGIQA-1k-Database-main/AIGC_MOS_Zscore.xlsx")
    df1.columns = [c.lower() for c in df1.columns]
    df1 = df1.rename(columns={df1.columns[0]: "filename", df1.columns[1]: "score"})
    df1["image_path"] = df1["filename"].astype(str).apply(
        lambda x: map1k.get(x.lower()) or map1k.get(Path(x).stem.lower()))
    df1["source"] = 0

    df2 = pd.read_csv(BASE / "AGIQA-3k-Database-main/AGIQA-3k-Database-main/data.csv")
    df2.columns = [c.lower() for c in df2.columns]
    df2 = df2.rename(columns={
        [c for c in df2.columns if "name" in c][0]: "filename",
        [c for c in df2.columns if "mos"  in c][0]: "score"})
    df2["image_path"] = df2["filename"].astype(str).apply(
        lambda x: map3k.get(x.lower()) or map3k.get(Path(x).stem.lower()))
    df2["source"] = 1

    df = pd.concat([df1, df2], ignore_index=True)
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
    df = df.dropna(subset=["image_path", "score"]).reset_index(drop=True)

    # ── KEY FIX: normalize scores per source so both datasets share the same scale
    for src in df["source"].unique():
        mask = df["source"] == src
        s = df.loc[mask, "score"]
        df.loc[mask, "score"] = (s - s.mean()) / (s.std() + 1e-8)

    print(f"Samples: {len(df)}  1K={( df.source==0).sum()}  3K={(df.source==1).sum()}")
    return df

df = load_data()

# ===== TRANSFORMS =====
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ===== DATASET =====
class IQA(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.tf = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        try:   img = Image.open(r["image_path"]).convert("RGB")
        except: img = Image.new("RGB", (224, 224))
        return self.tf(img), torch.tensor(r["score"], dtype=torch.float32)

# ===== MODELS =====
# Three diverse backbones for strong ensemble diversity
def make_effnet_b5():
    m = models.efficientnet_b5(weights="IMAGENET1K_V1")
    f = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(f, 512),
                                 nn.GELU(), nn.Dropout(0.2), nn.Linear(512, 1))
    return m

def make_convnext_small():
    m = models.convnext_small(weights="IMAGENET1K_V1")
    f = m.classifier[2].in_features
    m.classifier[2] = nn.Sequential(nn.Linear(f, 512), nn.GELU(),
                                    nn.Dropout(0.3), nn.Linear(512, 1))
    return m

def make_swin_t():
    m = models.swin_t(weights="IMAGENET1K_V1")
    f = m.head.in_features
    m.head = nn.Sequential(nn.Linear(f, 512), nn.GELU(),
                           nn.Dropout(0.3), nn.Linear(512, 1))
    return m

BACKBONES = [make_effnet_b5, make_convnext_small, make_swin_t]

# ===== LOSS =====
def pearson_loss(p, t):
    p, t = p.view(-1) - p.mean(), t.view(-1) - t.mean()
    return 1 - (p * t).sum() / (p.norm() * t.norm() + 1e-8)

def rank_loss(p, t):
    p, t = p.view(-1), t.view(-1)
    if len(p) < 2: return torch.tensor(0., device=p.device)
    return torch.relu(-(p.unsqueeze(1) - p.unsqueeze(0)) *
                       (t.unsqueeze(1) - t.unsqueeze(0))).mean()

def loss_fn(p, t):
    p, t = p.view(-1), t.view(-1)
    return (0.35 * nn.functional.mse_loss(p, t)
          + 0.35 * pearson_loss(p, t)
          + 0.30 * rank_loss(p, t))

# ===== MULTI-SCALE TTA =====
def tta_predict(model, x):
    sizes = [224, 192, 256]
    preds = []
    for s in sizes:
        xi = nn.functional.interpolate(x, size=(s, s), mode="bilinear", align_corners=False)
        preds.append(model(xi).view(-1))
        preds.append(model(torch.flip(xi, dims=[3])).view(-1))  # h-flip
    return torch.stack(preds).mean(0)

# ===== TRAIN ONE FOLD =====
def train_fold(model, tr_dl, epochs, lr):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = GradScaler()
    for ep in range(epochs):
        model.train()
        for x, y in tr_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with autocast():
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        sched.step()
    return model

# ===== OOF PREDICTIONS =====
def get_oof(seed):
    np.random.seed(seed); torch.manual_seed(seed)
    skf   = StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed)
    meta_y = df["score"].values.copy()
    meta_X = np.zeros((len(df), len(BACKBONES)))

    for b_id, backbone_fn in enumerate(BACKBONES):
        print(f"  Backbone {b_id+1}/{len(BACKBONES)}: {backbone_fn.__name__}")
        for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df["source"])):
            tr_dl = DataLoader(IQA(df.iloc[tr_idx], train_tf), BATCH,
                               shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
            va_dl = DataLoader(IQA(df.iloc[va_idx], val_tf),   BATCH,
                               shuffle=False, num_workers=2, pin_memory=True)

            model = train_fold(backbone_fn().to(DEVICE), tr_dl, EPOCHS, LR)

            model.eval()
            preds = []
            with torch.no_grad():
                for x, _ in va_dl:
                    preds.extend(tta_predict(model, x.to(DEVICE)).cpu().numpy())
            meta_X[va_idx, b_id] = preds
            print(f"    Fold {fold+1}  OOF SRCC={spearmanr(preds, meta_y[va_idx])[0]:.4f}")
            del model; torch.cuda.empty_cache()

    return meta_X, meta_y

# ===== META-LEARNER =====
def meta_blend(meta_X, meta_y):
    # Build rich cross-model features
    cols = [meta_X]
    for i in range(meta_X.shape[1]):
        for j in range(i+1, meta_X.shape[1]):
            cols.append((meta_X[:, i] + meta_X[:, j]).reshape(-1, 1) / 2)
            cols.append(np.abs(meta_X[:, i] - meta_X[:, j]).reshape(-1, 1))
    X = np.hstack(cols)
    X = StandardScaler().fit_transform(X)

    # Ridge (reliable on small feature sets)
    ridge = Ridge(alpha=0.5).fit(X, meta_y)
    r_pred = ridge.predict(X)

    # LightGBM — silent, shallow, small
    lgbm = lgb.LGBMRegressor(
        n_estimators=150, learning_rate=0.05, max_depth=3,
        num_leaves=7, min_child_samples=30, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=0.5,
        verbose=-1, random_state=42
    ).fit(X, meta_y)
    l_pred = lgbm.predict(X)

    blended = 0.5 * r_pred + 0.5 * l_pred
    print(f"  Ridge SRCC={spearmanr(r_pred,  meta_y)[0]:.4f} | "
          f"LGBM SRCC={spearmanr(l_pred,   meta_y)[0]:.4f} | "
          f"Blend SRCC={spearmanr(blended, meta_y)[0]:.4f}")
    return blended

# ===== MULTI-SEED RUN =====
all_preds = []
for seed in SEEDS:
    print(f"\n{'='*50}  seed={seed}  {'='*50}")
    meta_X, meta_y = get_oof(seed)
    preds = meta_blend(meta_X, meta_y)
    all_preds.append(preds)

final_preds = np.mean(all_preds, axis=0)
print(f"\n🔥 FINAL SRCC: {spearmanr(final_preds, df['score'].values)[0]:.4f}")

pd.DataFrame({"filename": df["filename"], "predicted_score": final_preds})\
  .to_csv("submission.csv", index=False)
print("🚀 submission.csv saved")

Device: cuda
Samples: 2982  1K=0  3K=2982

==================================================  seed=42  ==================================================
  Backbone 1/3: make_effnet_b5
Downloading: "https://download.pytorch.org/models/efficientnet_b5_lukemelas-1a07897c.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b5_lukemelas-1a07897c.pth


100%|██████████| 117M/117M [00:00<00:00, 132MB/s]  


    Fold 1  OOF SRCC=0.8491
    Fold 2  OOF SRCC=0.7963
    Fold 3  OOF SRCC=0.8540
    Fold 4  OOF SRCC=0.7837
    Fold 5  OOF SRCC=0.8109
  Backbone 2/3: make_convnext_small
Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /root/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


100%|██████████| 192M/192M [00:01<00:00, 137MB/s]  


    Fold 1  OOF SRCC=0.8751
    Fold 2  OOF SRCC=0.8257
    Fold 3  OOF SRCC=0.8607
    Fold 4  OOF SRCC=0.8250
    Fold 5  OOF SRCC=0.8494
  Backbone 3/3: make_swin_t
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 219MB/s] 


    Fold 1  OOF SRCC=0.8503
    Fold 2  OOF SRCC=0.8104
    Fold 3  OOF SRCC=0.8492
    Fold 4  OOF SRCC=0.7873
    Fold 5  OOF SRCC=0.8320
  Ridge SRCC=0.8528 | LGBM SRCC=0.8688 | Blend SRCC=0.8612

==================================================  seed=123  ==================================================
  Backbone 1/3: make_effnet_b5
    Fold 1  OOF SRCC=0.8095
    Fold 2  OOF SRCC=0.8306
    Fold 3  OOF SRCC=0.8340
    Fold 4  OOF SRCC=0.8129
    Fold 5  OOF SRCC=0.8488
  Backbone 2/3: make_convnext_small
    Fold 1  OOF SRCC=0.8267
    Fold 2  OOF SRCC=0.8615
    Fold 3  OOF SRCC=0.8489
    Fold 4  OOF SRCC=0.8327
    Fold 5  OOF SRCC=0.8601
  Backbone 3/3: make_swin_t
    Fold 1  OOF SRCC=0.8117
    Fold 2  OOF SRCC=0.8489
    Fold 3  OOF SRCC=0.8308
    Fold 4  OOF SRCC=0.8278
    Fold 5  OOF SRCC=0.8382
  Ridge SRCC=0.8517 | LGBM SRCC=0.8659 | Blend SRCC=0.8590

==================================================  seed=999  ==================================================